# The Bottleneck Viewpoint of $\mathcal W_\infty$

The distance $\Wass_\infty$ minimizes the largest displacement used by a coupling, rather than an average displacement.  This notebook builds a tiny equal-weight matching problem where the quadratic assignment and the bottleneck assignment differ, and highlights the longest transported edge in each case.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "notebooks-figures").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import (
    RED, BLUE, VIOLET, ORANGE, GRAY, LIGHT_GRAY,
    DIRAC_MARKER_SIZE, setup_matplotlib, figure_dir, save_pdf,
    remove_axes, box_axes, padded_limits, interp_color, draw_point_clouds,
)

setup_matplotlib()

from itertools import permutations
from scipy.optimize import linear_sum_assignment

NAME = "kantorovich-wasserstein-infinity"
OUT = figure_dir(NAME)

We search deterministically for a readable small example.  The bottleneck matching is found by exhaustive enumeration, which is harmless for seven points.

In [ ]:
def find_example():
    for seed in range(2000):
        rng = np.random.default_rng(seed)
        n = 7
        x = rng.normal(size=(n, 2)) @ np.array([[0.70, 0.15], [0.00, 0.44]]) + np.array([-0.45, 0.0])
        y = rng.normal(size=(n, 2)) @ np.array([[0.67, -0.08], [0.00, 0.50]]) + np.array([0.48, 0.03])
        x[0] += np.array([-0.50, 0.55])
        y[0] += np.array([0.55, -0.42])
        D = np.linalg.norm(x[:, None, :] - y[None, :, :], axis=2)
        _, col = linear_sum_assignment(D**2)
        perm2 = tuple(col)
        cost2 = (D[np.arange(n), col]**2).sum()
        max2 = D[np.arange(n), col].max()
        best_perm = None
        best_max = np.inf
        best_sum = np.inf
        for perm in permutations(range(n)):
            vals = D[np.arange(n), perm]
            m = vals.max()
            s = (vals**2).sum()
            if (m < best_max - 1e-12) or (abs(m - best_max) < 1e-12 and s < best_sum):
                best_max = m
                best_sum = s
                best_perm = perm
        if best_perm != perm2 and max2 > best_max * 1.10:
            return x, y, perm2, best_perm, D
    raise RuntimeError("No example found")

x, y, perm2, perminf, D = find_example()

Both panels use the same point clouds.  Violet segments encode the matching; the orange segment is the maximal displacement that controls $\Wass_\infty$.

In [ ]:
def draw_matching(perm, path):
    fig, ax = plt.subplots(figsize=(2.25, 2.25))
    pts = np.vstack([x, y])
    (xmin, xmax), (ymin, ymax) = padded_limits(pts, pad=0.28)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal")
    remove_axes(ax)
    lengths = D[np.arange(len(x)), perm]
    imax = int(np.argmax(lengths))
    for i, j in enumerate(perm):
        color = ORANGE if i == imax else VIOLET
        lw = 2.1 if i == imax else 0.75
        alpha = 0.95 if i == imax else 0.55
        ax.plot([x[i,0], y[j,0]], [x[i,1], y[j,1]], color=color, lw=lw, alpha=alpha, zorder=1)
    draw_point_clouds(ax, x, y, base_size=DIRAC_MARKER_SIZE)
    save_pdf(fig, OUT / path, pad_inches=0.06)
    plt.close(fig)

draw_matching(perm2, "quadratic.pdf")
draw_matching(perminf, "bottleneck.pdf")